### Bronze Layer | Structure Streaming - AutoLoader



- The Bronze layer serves as the raw ingestion layer and long-term historical archive.

- On each run, Auto Loader uses checkpoints to compare incoming files against those already processed, appending only new files — ensuring no data is ever processed twice or lost as the 100-day API window shifts forward over time.

- **Note:** Landing files into the volume follows a classic watermark approach, saving files only for dates not yet ingested.

In [0]:
%run ../notebooks/config_Parms

In [0]:
df= spark.read.parquet("/Volumes/workspace/bronze/landing_zone/ibm_landing")
df.printSchema()


In [0]:
# read schema and declareared in Config_parms


# Auto Loader: Stream reads from landing zone
stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schemas/ibm_stream") \
    .schema(schema) \
    .load("/Volumes/workspace/bronze/landing_zone/ibm_landing") \
    .withColumn("ingested_at", current_timestamp())

# Write stream to Delta table
query = stream_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/ibm_stream") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .table(f"{catalog}.{bronze_schema}.ibm")

# Check stream status
print(query.status)

In [0]:
df=spark.table(f"{catalog}.{bronze_schema}.ibm")
df.display()


dbutils.fs.ls("dbfs:/Volumes/workspace/bronze/landing_zone/")